<a href="https://colab.research.google.com/github/YasserJEP/TDA-BifurcacionesHopf/blob/main/Notebooks/2_Norma_L1_Vector_Betti_(H_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hopf Normal

Este notebook calcula la norma L1 de la curva de Betti-1 (H₁) para el sistema Hopf normal con μ ∈ [-1,1] (300 valores). Usa embedding de Takens (d=2, τ=26, stride=2) y Vietoris-Rips con métrica Manhattan. Sin ruido. Exporta gráfico PDF y resultados a Excel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
from gtda.time_series import SingleTakensEmbedding
from gtda.homology import VietorisRipsPersistence
import teaspoon.MakeData.DynSysLib.DynSysLib as DSL

# ============================================================
#  EMBEDDING DE TAKENS
# ============================================================
def getTakensEmbedding(x, dim, Tau, dT=2):
    """
    Embedding de Takens usando giotto-tda.
    dim: dimensión del embedding
    Tau: tiempo de retardo
    dT: stride (salto entre ventanas)
    """
    embedder = SingleTakensEmbedding(
        parameters_type="fixed",
        time_delay=Tau,
        dimension=dim,
        stride=dT,
        n_jobs=4
    )
    X = embedder.fit_transform(x)
    return X

ruido = 0.00

# ------------------------- SERIE TEMPORAL HOPF NORMAL ------------------------- #
def generate_Hopf_Normal_series(mu, sample_size=2000, fs=50):
    """Genera serie temporal x(t) del sistema Hopf normal usando DynSysLib."""
    parameters = [mu, 6]  # [mu, omega]
    initial_conditions = [0, 1.01]

    _, ts = DSL.DynamicSystems(
        system="hopf_normal",
        dynamic_state=None,
        L=50,
        fs=100,
        SampleSize=2000,
        UserGuide=False,
        parameters=parameters,
        InitialConditions=initial_conditions
    )

    def add_noise(ts, noise_level=0.1, mode="relative", seed=None):
        if seed is not None:
            np.random.seed(seed)
        ts = np.asarray(ts)
        if mode == "relative":
            scale = noise_level * np.std(ts, axis=-1, keepdims=True)
        else:
            scale = noise_level
        noise = scale * np.random.randn(*ts.shape)
        return ts + noise

    ts = add_noise(ts, noise_level=ruido, mode="relative", seed=42)
    return ts[0]  # coordenada x(t)

# ------------------------- BETTI 1 ------------------------- #
max_edge = 20

def compute_betti_norm_1(point_cloud, max_edge=max_edge, n_samples=100):
    VR = VietorisRipsPersistence(
        homology_dimensions=[1],
        metric="manhattan",
        max_edge_length=max_edge
    )

    diagrams = VR.fit_transform(point_cloud[None, :, :])
    diag_H1 = diagrams[0]
    diag_H1 = diag_H1[np.isfinite(diag_H1[:, 1])]

    P = np.linspace(0, max_edge, n_samples)
    betti_1 = np.zeros(n_samples)

    for i, e in enumerate(P):
        betti_1[i] = np.sum((diag_H1[:, 0] <= e) & (e < diag_H1[:, 1]))

    return np.sum(betti_1)

dim = 2
tau = 26
dT = 2

# ======================================================================
# FUNCIÓN PRINCIPAL
# ======================================================================
def compute_L1_for_mu(mu, dim=dim, Tau=tau, dT=dT):
    """Calcula la norma L1 de Bv_1 usando embedding de Takens."""
    serie_x = generate_Hopf_Normal_series(mu)
    embedding = getTakensEmbedding(serie_x, dim=dim, Tau=tau, dT=dT)
    return compute_betti_norm_1(embedding, max_edge=max_edge)

# ======================================================================
# GRÁFICA PRINCIPAL
# ======================================================================
def plot_betti_norm_1_Hopf_Normal(mu_values, dim=dim, Tau=tau, dT=dT):
    """Calcula y grafica la norma L1 de Bv_1 para distintos valores de μ."""
    mu_vals = []
    L1_norms_vals = []

    for i, mu in enumerate(mu_values):
        l1 = compute_L1_for_mu(mu, dim, Tau, dT)
        L1_norms_vals.append(l1)
        mu_vals.append(mu)
        print(f"μ = {mu:.5f} → ‖Bv₁‖₁ = {l1:.4f} ({i+1}/{len(mu_values)})")

    plt.figure(figsize=(10, 3))
    plt.plot(mu_values, L1_norms_vals, 'r', lw=1.2)
    plt.xlim(-1, 1)
    plt.xticks(np.arange(-1, 1.1, 0.25))
    plt.margins(x=0.01)
    plt.xlabel('$\\mu$', fontsize=12)
    plt.ylabel('$||Bv_1||_1$', fontsize=12)
    plt.axvline(0.0, linestyle='--', lw=1, alpha=0.9, c="black", label=r"$\mu_c = 0$")
    plt.grid(True, alpha=0.2)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"figure_L1_norms_Hopf_Normal_n{len(mu_values)}_Tau{tau}_m{dim}_noise_{ruido}.pdf",
                bbox_inches="tight", dpi=600)
    plt.show()

    # Guardar resultados en Excel
    carpeta = r"D:\MAESTRÍA\CÓDIGOS TDA-BIFURCACIONES DE HOPF\6_NORMA L1 DE LOS VECTORES DE BETTI\HOPF NORMAL"
    os.makedirs(carpeta, exist_ok=True)

    archivo_excel = os.path.join(carpeta, f"resultados_L1_Hopf_Normal_{len(mu_values)}_Tau={tau}_m={dim}_ruido={ruido}.xlsx")
    df = pd.DataFrame({"mu": mu_vals, "L1_norm": L1_norms_vals})
    df.to_excel(archivo_excel, index=False)
    print(f"\nArchivo Excel guardado en:\n{archivo_excel}\n")

# ============================================================
# EJECUCIÓN
# ============================================================
if __name__ == "__main__":
    mu_values = np.round(np.linspace(-1, 1, 300), 5)
    plot_betti_norm_1_Hopf_Normal(mu_values, dim=dim, Tau=tau, dT=dT)

# Lorenz

Este notebook calcula la norma L1 de la curva de Betti-1 (H₁) para el sistema Lorenz con ρ ∈ [20,30] (300 valores). Usa embedding de Takens (d=2, τ=16, stride=2) y Vietoris-Rips con métrica Manhattan. Sin ruido. Exporta gráfico PDF y resultados a Excel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from gtda.time_series import SingleTakensEmbedding
from gtda.homology import VietorisRipsPersistence
import teaspoon.MakeData.DynSysLib.DynSysLib as DSL

# ============================================================
#  EMBEDDING DE TAKENS
# ============================================================
def getTakensEmbedding(x, dim, Tau, dT=2):
    """
    Embedding de Takens usando giotto-tda.
    dim: dimensión del embedding
    Tau: tiempo de retardo
    dT: stride (salto entre ventanas)
    """
    embedder = SingleTakensEmbedding(
        parameters_type="fixed",
        time_delay=Tau,
        dimension=dim,
        stride=dT,
        n_jobs=4
    )
    X = embedder.fit_transform(x)
    return X

ruido = 0.0

# ------------------------- SERIE TEMPORAL LORENZ ------------------------- #
def generate_lorenz_series(rho, sample_size=2000, fs=130):
    """Genera serie temporal x(t) del sistema Lorenz usando DynSysLib."""
    parameters = [rho, 10, 8 / 3]  # [rho, sigma, beta]
    initial_conditions = [6.74, 6.74, 22.74]

    _, ts = DSL.DynamicSystems(
        system="lorenz",
        dynamic_state=None,
        L=100,
        fs=130,
        SampleSize=2000,
        UserGuide=False,
        parameters=parameters,
        InitialConditions=initial_conditions
    )

    def add_noise(ts, noise_level=0.1, mode="relative", seed=None):
        if seed is not None:
            np.random.seed(seed)
        ts = np.asarray(ts)
        if mode == "relative":
            scale = noise_level * np.std(ts, axis=-1, keepdims=True)
        else:
            scale = noise_level
        noise = scale * np.random.randn(*ts.shape)
        return ts + noise

    ts = add_noise(ts, noise_level=ruido, mode="relative", seed=42)
    return ts[0]  # coordenada x(t)

# ------------------------- BETTI 1 ------------------------- #
max_edge = 20

def compute_betti_norm_1(point_cloud, max_edge=max_edge, n_samples=40):
    VR = VietorisRipsPersistence(
        homology_dimensions=[1],
        metric="manhattan",
        max_edge_length=max_edge
    )

    diagrams = VR.fit_transform(point_cloud[None, :, :])
    diag_H1 = diagrams[0]
    diag_H1 = diag_H1[np.isfinite(diag_H1[:, 1])]

    P = np.linspace(0, max_edge, n_samples)
    betti_1 = np.zeros(n_samples)

    for i, e in enumerate(P):
        betti_1[i] = np.sum((diag_H1[:, 0] <= e) & (e < diag_H1[:, 1]))

    return np.sum(betti_1)

dim = 2
Tau = 16
dT = 2

# ======================================================================
# FUNCIÓN PRINCIPAL
# ======================================================================
def compute_L1_for_rho(rho, dim=dim, Tau=Tau, dT=dT):
    """Calcula la norma L1 de Bv_1 usando embedding de Takens."""
    serie_x = generate_lorenz_series(rho)
    embedding = getTakensEmbedding(serie_x, dim=dim, Tau=Tau, dT=dT)
    return compute_betti_norm_1(embedding, max_edge=max_edge)

# ======================================================================
# GRÁFICA PRINCIPAL
# ======================================================================
def plot_betti_norm_1_lorenz(rho_values, dim=dim, Tau=Tau, dT=dT):
    """Calcula y grafica la norma L1 de Bv_1 para distintos valores de ρ."""
    rho_vals = []
    L1_norms_vals = []

    for i, rho in enumerate(rho_values):
        l1 = compute_L1_for_rho(rho, dim, Tau, dT)
        L1_norms_vals.append(l1)
        rho_vals.append(rho)
        print(f"ρ = {rho:.5f} → ‖Bv₁‖₁ = {l1:.5f} ({i+1}/{len(rho_values)})")

    # Guardar resultados en Excel
    carpeta = r"D:\MAESTRÍA\CÓDIGOS TDA-BIFURCACIONES DE HOPF\6_NORMA L1 DE LOS VECTORES DE BETTI\LORENZ"
    os.makedirs(carpeta, exist_ok=True)

    archivo_excel = os.path.join(carpeta, f"resultados_L1_Lorenz_{len(rho_values)}_Tau={Tau}_m={dim}_noise={ruido}.xlsx")
    df = pd.DataFrame({"rho": rho_vals, "L1_norm": L1_norms_vals})
    df.to_excel(archivo_excel, index=False)
    print(f"\nArchivo Excel guardado en:\n{archivo_excel}\n")

    plt.figure(figsize=(10, 3))
    plt.plot(rho_values, L1_norms_vals, 'r', lw=1.2)
    plt.xlim(20, 30)
    plt.xticks(np.arange(20, 30.1, 1))
    plt.margins(x=0.01)
    plt.xlabel('$\\rho$', fontsize=12)
    plt.ylabel('$||Bv_1||_1$', fontsize=12)
    plt.grid(True, alpha=0.1)
    plt.axvline(24.74, linestyle='--', lw=1, alpha=0.9, c="black", label=r"$\rho_c \approx$ 24.74")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"figure_L1_Lorenz_n={len(rho_values)}_Tau={Tau}_m={dim}_noise={ruido}.pdf",
                bbox_inches="tight", dpi=600)
    plt.show()

# ============================================================
# EJECUCIÓN
# ============================================================
if __name__ == "__main__":
    rho_values = np.round(np.linspace(20, 30, 300), 5)
    plot_betti_norm_1_lorenz(rho_values, dim=dim, Tau=Tau, dT=dT)

# Reacción BZ

Este notebook calcula la norma L1 de la curva de Betti-1 (H₁) para el oscilador de Belousov-Zhabotinsky con b ∈ [2,5] (300 valores). Usa embedding de Takens (d=2, τ=57, stride=5) y Vietoris-Rips con métrica Manhattan. Exporta gráfico PDF y resultados a Excel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gtda.time_series import SingleTakensEmbedding
from gtda.homology import VietorisRipsPersistence
import teaspoon.MakeData.DynSysLib.DynSysLib as DSL
import pandas as pd
import os

# ============================================================
# PARÁMETROS GLOBALES DEL EMBEDDING DE TAKENS
# ============================================================
TAKENS_PARAMS = {
    "dim": 2,
    "Tau": 57,
    "dT": 5,
    "n_jobs": 4
}

# ============================================================
# EMBEDDING DE TAKENS
# ============================================================
def getTakensEmbedding(x, params):
    """
    Embedding de Takens usando giotto-tda.
    """
    embedder = SingleTakensEmbedding(
        parameters_type="fixed",
        time_delay=params["Tau"],
        dimension=params["dim"],
        stride=params["dT"],
        n_jobs=params["n_jobs"]
    )
    return embedder.fit_transform(x)

L = 80
fs = 90
SampleSize = 4000

# ============================================================
# SERIE TEMPORAL – MODELO BELUSOV–ZHABOTINSKY (2D)
# ============================================================
def generate_BZ_series(b, sample_size=SampleSize):
    parameters = [10, b]  # [a, b]
    initial_conditions = [1.0, 5.01]

    _, ts = DSL.DynamicSystems(
        system="oscilador_Belousov_Zhabotinski_2D",
        dynamic_state=None,
        L=L,
        fs=fs,
        SampleSize=SampleSize,
        UserGuide=False,
        parameters=parameters,
        InitialConditions=initial_conditions
    )
    return ts[0]

# ============================================================
# NORMA L1 DEL VECTOR DE BETTI-1
# ============================================================
max_edge = 40

def compute_betti_norm_1(point_cloud, max_edge=max_edge, n_samples=80):
    VR = VietorisRipsPersistence(
        homology_dimensions=[1],
        metric="manhattan",
        max_edge_length=max_edge
    )

    diagrams = VR.fit_transform(point_cloud[None, :, :])
    diag_H1 = diagrams[0]
    diag_H1 = diag_H1[np.isfinite(diag_H1[:, 1])]

    P = np.linspace(0, max_edge, n_samples)
    betti_1 = np.zeros(n_samples)

    for i, e in enumerate(P):
        betti_1[i] = np.sum((diag_H1[:, 0] <= e) & (e < diag_H1[:, 1]))

    return np.sum(betti_1)

# ============================================================
# CÁLCULO PARA UN VALOR DE b
# ============================================================
def compute_L1_for_b(b, takens_params):
    serie_x = generate_BZ_series(b)
    embedding = getTakensEmbedding(serie_x, takens_params)
    return compute_betti_norm_1(embedding)

# ============================================================
# FUNCIÓN PRINCIPAL DE ANÁLISIS Y GRÁFICA
# ============================================================
def plot_betti_norm_1_BZ(b_values, takens_params):
    L1_norms = []

    for i, b in enumerate(b_values):
        l1 = compute_L1_for_b(b, takens_params)
        L1_norms.append(l1)
        print(f"b = {b:.5f} → ‖Bv₁‖₁ = {l1:.6f} ({i+1}/{len(b_values)})")

    # Guardar resultados
    carpeta = r"D:\MAESTRÍA\CÓDIGOS TDA-BIFURCACIONES DE HOPF\6_NORMA L1 DE LOS VECTORES DE BETTI\REACCIÓN BZ"
    os.makedirs(carpeta, exist_ok=True)

    archivo_excel = os.path.join(
        carpeta,
        f"resultados_L1_BZ_dim={takens_params['dim']}_Tau={takens_params['Tau']}_dT={takens_params['dT']}_b=({b_values[0]}, {b_values[-1]})_N={len(b_values)}.xlsx"
    )

    df = pd.DataFrame({"b": b_values, "L1_norm": L1_norms})
    df.to_excel(archivo_excel, index=False)
    print(f"\nArchivo Excel guardado en:\n{archivo_excel}\n")

    # Gráfica
    plt.figure(figsize=(10, 3))
    plt.plot(b_values, L1_norms, 'r', lw=1.2)
    plt.xlim(2, 5)
    plt.xticks(np.arange(2, 5.1, 0.5))
    plt.margins(x=0.01)
    plt.xlabel('$b$', fontsize=13)
    plt.ylabel('$||Bv_1||_1$', fontsize=13)
    plt.grid(alpha=0.06)
    plt.axvline(3.5, linestyle='--', lw=1, alpha=0.9, c="black", label=r"$b_c = 3.5$")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"figure_L1_BZ_dim={takens_params['dim']}_Tau={takens_params['Tau']}_dT={takens_params['dT']}_b=({b_values[0]}, {b_values[-1]})_N={len(b_values)}.pdf",
                bbox_inches="tight", dpi=600)
    plt.show()

# ============================================================
# EJECUCIÓN
# ============================================================
if __name__ == "__main__":
    b_values = np.round(np.linspace(2, 5, 300), 5)
    plot_betti_norm_1_BZ(b_values=b_values, takens_params=TAKENS_PARAMS)